In [2]:
!pip install folium

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.4/113.4 kB 4.0 MB/s eta 0:00:00


In [3]:
import pandas as pd
import folium
from folium import plugins

In [6]:
def create_city_map(csv_file, output_file='city_statistics_map.html'):
    """
    Create an interactive map with proportionate circles based on city counts.
    Includes U.S. state boundaries as a base layer.
    
    Args:
        csv_file: Path to CSV file with columns: city, count, state, latitude, longitude
        output_file: Path to save the HTML map
    """
    # Read the CSV file
    df = pd.read_csv(csv_file)
    
    # Remove rows with missing coordinates
    df = df.dropna(subset=['latitude', 'longitude'])
    
    print(f"Loaded {len(df)} cities with valid coordinates")
    
    # Calculate map center
    center_lat = df['latitude'].mean()
    center_lon = df['longitude'].mean()
    
    # Create base map
    m = folium.Map(
        location=[center_lat, center_lon],
        zoom_start=4,
        tiles='CartoDB positron',
        control_scale=True
    )
    
    # Add U.S. state boundaries
    state_geo_url = 'https://raw.githubusercontent.com/PublicaMundi/MappingAPI/master/data/geojson/us-states.json'
    
    folium.GeoJson(
        state_geo_url,
        name='U.S. State Boundaries',
        style_function=lambda feature: {
            'fillColor': '#f0f0f0',
            'color': '#666666',
            'weight': 1.5,
            'fillOpacity': 0.15
        },
        highlight_function=lambda feature: {
            'fillColor': '#e0e0e0',
            'color': '#333333',
            'weight': 2,
            'fillOpacity': 0.3
        },
        tooltip=folium.GeoJsonTooltip(fields=['name'], aliases=['State:'])
    ).add_to(m)
    
    # Calculate radius scaling
    max_count = df['count'].max()
    min_count = df['count'].min()
    
    def scale_radius(count):
        """Scale count to appropriate circle radius in meters"""
        if max_count == min_count:
            return 50000
        normalized = (count - min_count) / (max_count - min_count)
        min_radius = 20000
        max_radius = 150000
        return min_radius + (normalized ** 0.5) * (max_radius - min_radius)
    
    def get_color(count):
        """Get color based on count value"""
        normalized = (count - min_count) / (max_count - min_count) if max_count != min_count else 0.5
        
        if normalized < 0.33:
            return '#3b82f6'
        elif normalized < 0.67:
            return '#f59e0b'
        else:
            return '#ef4444'
    
    # Add circles for each city
    for idx, row in df.iterrows():
        radius = scale_radius(row['count'])
        color = get_color(row['count'])
        
        state_text = row['state'] if pd.notna(row['state']) else 'N/A'
        
        popup_html = f"""
        <div style="font-family: Arial, sans-serif; min-width: 150px;">
            <h4 style="margin: 0 0 10px 0; color: #1e293b;">{row['city']}</h4>
            <table style="width: 100%; font-size: 13px;">
                <tr>
                    <td style="padding: 3px 5px; color: #64748b;">State:</td>
                    <td style="padding: 3px 5px; font-weight: bold;">{state_text}</td>
                </tr>
                <tr>
                    <td style="padding: 3px 5px; color: #64748b;">Count:</td>
                    <td style="padding: 3px 5px; font-weight: bold; color: {color};">{int(row['count'])}</td>
                </tr>
            </table>
        </div>
        """
        
        folium.Circle(
            location=[row['latitude'], row['longitude']],
            radius=radius,
            popup=folium.Popup(popup_html, max_width=250),
            tooltip=f"{row['city']}: {int(row['count'])}",
            color=color,
            fill=True,
            fillColor=color,
            fillOpacity=0.5,
            weight=2,
            opacity=0.8
        ).add_to(m)
        
        if row['count'] > max_count * 0.5:
            label_html = f"""
                <div style="
                    font-size: 11px;
                    font-weight: bold;
                    color: #1e293b;
                    text-align: center;
                    white-space: nowrap;
                    text-shadow: 1px 1px 2px white, -1px -1px 2px white, 1px -1px 2px white, -1px 1px 2px white;
                ">{row['city']}</div>
            """
            folium.Marker(
                location=[row['latitude'], row['longitude']],
                icon=folium.DivIcon(html=label_html)
            ).add_to(m)
    
    # Add legend
    legend_html = f"""
    <div style="
        position: fixed;
        bottom: 50px;
        right: 50px;
        width: 220px;
        background-color: white;
        border: 2px solid #666;
        border-radius: 5px;
        padding: 15px;
        font-family: Arial, sans-serif;
        font-size: 13px;
        box-shadow: 0 2px 10px rgba(0,0,0,0.2);
        z-index: 1000;
    ">
        <h4 style="margin: 0 0 10px 0; font-size: 14px; color: #1e293b;">Legend</h4>
        <div style="margin-bottom: 8px;">
            <strong>Circle Size:</strong> Proportional to count
        </div>
        <div style="margin-bottom: 5px;">
            <span style="display: inline-block; width: 12px; height: 12px; background-color: #3b82f6; border-radius: 50%; margin-right: 5px;"></span>
            Low (1-{int(max_count * 0.33)})
        </div>
        <div style="margin-bottom: 5px;">
            <span style="display: inline-block; width: 12px; height: 12px; background-color: #f59e0b; border-radius: 50%; margin-right: 5px;"></span>
            Medium ({int(max_count * 0.33)}-{int(max_count * 0.67)})
        </div>
        <div>
            <span style="display: inline-block; width: 12px; height: 12px; background-color: #ef4444; border-radius: 50%; margin-right: 5px;"></span>
            High ({int(max_count * 0.67)}+)
        </div>
        <div style="margin-top: 10px; padding-top: 10px; border-top: 1px solid #e5e7eb; font-size: 11px; color: #64748b;">
            Total cities: {len(df)}<br>
            Max count: {int(max_count)}<br>
            Min count: {int(min_count)}
        </div>
    </div>
    """
    m.get_root().html.add_child(folium.Element(legend_html))
    
    folium.LayerControl().add_to(m)
    
    plugins.Fullscreen(
        position='topleft',
        title='Fullscreen',
        title_cancel='Exit fullscreen',
        force_separate_button=True
    ).add_to(m)
    
    m.save(output_file)
    print(f"\nMap saved to: {output_file}")
    print(f"Open the file in a web browser to view the interactive map.")
    
    print(f"\nSummary Statistics:")
    print(f"  Total cities: {len(df)}")
    print(f"  Max count: {int(max_count)} ({df.loc[df['count'].idxmax(), 'city']})")
    print(f"  Min count: {int(min_count)}")
    print(f"  Average count: {df['count'].mean():.1f}")
    print(f"  Total count: {int(df['count'].sum())}")

In [7]:
def main():
    input_file = 'city_statistics_with_coords.csv'
    output_file = 'city_statistics_map.html'
    
    print("="*60)
    print("City Statistics Map Generator")
    print("="*60)
    print()
    
    try:
        create_city_map(input_file, output_file)
    except FileNotFoundError:
        print(f"Error: Could not find '{input_file}'")
        print("Please make sure the file exists in the current directory.")
        print("Expected columns: city, count, state, latitude, longitude")
    except Exception as e:
        print(f"Error: {e}")
        import traceback
        traceback.print_exc()

In [8]:
if __name__ == '__main__':
    main()

City Statistics Map Generator

Loaded 24 cities with valid coordinates

Map saved to: city_statistics_map.html
Open the file in a web browser to view the interactive map.

Summary Statistics:
  Total cities: 24
  Max count: 11 (Washington)
  Min count: 1
  Average count: 1.9
  Total count: 46
